# 17 - Cuantos frames del pool ve el modelo por paso

## El problema

Cada paso de entrenamiento alimenta al modelo con **30 imagenes etiquetadas** (5 items,
cada uno con 1 vista base + 5 aumentadas) y **1 solo frame sin etiquetar**, porque el
codigo hace `batch_size // 4 = 1`.

FixMatch usa **7 sin etiquetar por cada etiquetado**. Aqui la proporcion real es
**1 por cada 30**.

## Que se cambia

Solo `batch_size_unlab`, de 1 a 4. **El batch etiquetado sigue en 5 items**, para que la
comparacion contra los runs que ya existen sea limpia.

## Los dos pools, y por que se prueban los dos

| Pool | Frames | Hoy ve | Con 4 veria |
|---|---|---|---|
| `r10_max0` | 3.937 | ~80 % del pool, una vez | 100 %, ~3 veces cada frame |
| `all_lateral` | 74.774 | **~4 %** | **~17 %** |

En `r10` el margen es pequeno: ya recorre casi todo el pool, asi que darle 4 sobre todo
**repite** frames. En `all_lateral` el margen es enorme: hoy ni se acerca a recorrerlo, y
con 4 ve **cuatro veces mas frames distintos**.

Por eso se prueban los dos: distinguen **dos causas diferentes**.

- Si mejora **solo en all_lateral** -> lo que faltaba eran **frames nuevos**.
- Si mejora **en los dos** -> lo que faltaba era **senal de la perdida SSL por paso**,
  no cobertura del pool.
- Si no mejora en ninguno -> el limite no estaba aqui, y eso **refuerza** la conclusion
  actual de la tesis.

Los tres desenlaces son informativos.

## Contra que se compara

| Baseline | Semillas | F1 |
|---|---|---|
| `runs_final_v1/mean_teacher_r10` | 5 | 0.8502 +/- 0.0065 |
| `runs_final_v1/mean_teacher_all_lateral` | 3 | 0.8602 +/- 0.0058 |

El segundo es **el mejor resultado de la tesis** (el 0.860 de la tabla).

## Coste

Un run tarda **~30 minutos** (21,4 s/epoca, ~75 epocas). Seis runs = **~3 horas**, una
sesion de Colab. La memoria sube poco: de 31 a 34 imagenes con gradiente por paso, +10 %.

## Orden de ejecucion

SETUP, luego VERIFICACION (2 min, no entrena), luego las 6 celdas de entrenamiento,
luego RESUMEN.

Si diera `CUDA out of memory`, bajar `batch_size_unlab` a 2 en todas las celdas y
anotarlo, porque cambia el experimento.


In [ ]:
# ============================================================
# SETUP - correr una vez tras cada reinicio del runtime
# No entrena nada.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"

import sys
sys.path.append("/content/tesis-seg")

import json, os, time, glob

from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.datasets import (
    build_supervised_datasets,
    build_unlabeled_datasets,
    build_dataloaders,
)
from src.train import run_training
from src.evaluate import evaluate_checkpoint

# --- GUARDIAN: el codigo clonado debe leer batch_size_unlab del config ---
# El 2026-08-16 se perdieron 20 runs porque un flag se ignoro en silencio.
# Comprobacion estatica sobre el fuente clonado. No entrena nada.
import inspect
from src import datasets as _ds
_src = inspect.getsource(_ds.build_dataloaders)
assert "batch_size_unlab" in _src, (
    "CODIGO VIEJO: build_dataloaders no lee batch_size_unlab. "
    "Borra /content/tesis-seg, vuelve a clonar y reinicia el entorno."
)
print("OK: el codigo clonado lee batch_size_unlab del config.")


---
### Paso 1 - Verificar ANTES de gastar GPU


In [ ]:
# ============================================================
# VERIFICACION - NO ENTRENA. Tarda ~2 minutos.
# Comprueba que batch_size_unlab=4 llega de verdad al DataLoader,
# en los DOS pools, y calcula cuanto pool se ve.
# CORRER ESTA CELDA ANTES QUE NINGUNA DE ENTRENAMIENTO.
# ============================================================
EPOCAS_TIPICAS = 75   # mediana observada en los runs de UNM

base = get_default_config()
base["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
base["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
base["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

base["arch"] = "unetpp"; base["backbone"] = "efficientnet-b3"; base["n_classes"] = 1
base["use_semi"] = True
base["ssl_method"] = "mean_teacher"
base["target_size"] = (320, 320); base["use_pad"] = True; base["use_fixed_crop"] = False
base["imagenet_norm"] = False; base["image_preproc"] = "base"; base["mask_smoothing"] = "none"
base["num_augmented"] = 5; base["num_workers"] = 4; base["drop_last"] = True

train_tf  = get_supervised_train_augmentation(base)
weak_tf   = get_weak_augmentation(base)
strong_tf = get_strong_augmentation(base)

POOLS = [
    ("r10_max0",    "unlabeling_r10_max0/images"),
    ("all_lateral", "unlabeling_all_lateral/images"),
]

for nombre_pool, subdir in POOLS:
    print("=" * 62)
    print("POOL:", nombre_pool)
    print("=" * 62)
    for _bu in [None, 4]:
        c = dict(base)
        c["unlabeled_subdir"] = subdir
        c["batch_size"] = 5
        if _bu is not None:
            c["batch_size_unlab"] = _bu
        tr, va, te = build_supervised_datasets(c, train_tf=train_tf)
        uds, tds = build_unlabeled_datasets(c, weak_tf=weak_tf, strong_tf=strong_tf)
        L = build_dataloaders(c, train_ds=tr, val_ds=va, test_ds=te,
                              unlabeled_ds=uds, temporal_unlab_ds=tds)
        pasos   = len(L["train_loader"])
        b_unlab = L["unlabeled_loader"].batch_size
        pool    = len(uds)
        por_ep  = pasos * b_unlab
        total   = por_ep * EPOCAS_TIPICAS
        pasadas = total / float(pool)
        etiqueta = "ACTUAL (sin batch_size_unlab)" if _bu is None else "NUEVO  (batch_size_unlab=4)"
        print("  " + etiqueta)
        print("    batch etiquetado (items)   : %d  -> %d imagenes reales (1 base + 5 aumentadas)"
              % (L["train_loader"].batch_size, L["train_loader"].batch_size * 6))
        print("    batch NO etiquetado        : %d frames" % b_unlab)
        print("    pasos por epoca            : %d" % pasos)
        print("    pool total                 : %d frames" % pool)
        print("    visto por epoca            : %d frames (%.2f %%)" % (por_ep, 100.0 * por_ep / pool))
        print("    en %d epocas               : %d extracciones = %.2f pasadas por el pool"
              % (EPOCAS_TIPICAS, total, pasadas))
        if pasadas < 1:
            print("       -> NO llega a recorrer el pool entero ni una vez")
        print()
        if _bu == 4:
            assert b_unlab == 4, "FALLO: batch_size_unlab NO llego al DataLoader. NO entrenar."
            assert L["train_loader"].batch_size == 5, "FALLO: el batch etiquetado cambio. Debe seguir en 5."

print("OK: el cambio llega al DataLoader en los dos pools y el batch etiquetado sigue en 5.")
print("Ya puedes correr las celdas de entrenamiento.")


---
### Paso 2a - Pool `r10_max0` (3 semillas)

Baseline: `mean_teacher_r10`, 0.8502 +/- 0.0065 con 5 semillas.


In [ ]:
# === UNM Mean Teacher r10, batch_size_unlab=4, seed 0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_r10_unlab4"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5

# --- LA UNICA DIFERENCIA con mean_teacher_r10 ---
# Por defecto el codigo usa batch_size // 4 = 1 frame no etiquetado por paso.
# Con 4, el modelo ve cuatro veces mas frames del pool por epoca.
# El batch etiquetado NO se toca: sigue en 5 items (30 imagenes), para que la
# comparacion contra el baseline sea limpia.
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5, "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4, "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05, "lambda_u debe igualar al del baseline"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_r10_max0/images", "pool equivocado en esta celda"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    # guardian en caliente: el loader debe tener 4 de verdad
    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando para no "
        "gastar GPU en un run identico al viejo."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM Mean Teacher r10, batch_size_unlab=4, seed 1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_r10_unlab4"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5

# --- LA UNICA DIFERENCIA con mean_teacher_r10 ---
# Por defecto el codigo usa batch_size // 4 = 1 frame no etiquetado por paso.
# Con 4, el modelo ve cuatro veces mas frames del pool por epoca.
# El batch etiquetado NO se toca: sigue en 5 items (30 imagenes), para que la
# comparacion contra el baseline sea limpia.
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5, "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4, "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05, "lambda_u debe igualar al del baseline"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_r10_max0/images", "pool equivocado en esta celda"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    # guardian en caliente: el loader debe tener 4 de verdad
    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando para no "
        "gastar GPU en un run identico al viejo."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM Mean Teacher r10, batch_size_unlab=4, seed 2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_r10_unlab4"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5

# --- LA UNICA DIFERENCIA con mean_teacher_r10 ---
# Por defecto el codigo usa batch_size // 4 = 1 frame no etiquetado por paso.
# Con 4, el modelo ve cuatro veces mas frames del pool por epoca.
# El batch etiquetado NO se toca: sigue en 5 items (30 imagenes), para que la
# comparacion contra el baseline sea limpia.
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5, "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4, "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05, "lambda_u debe igualar al del baseline"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_r10_max0/images", "pool equivocado en esta celda"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    # guardian en caliente: el loader debe tener 4 de verdad
    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando para no "
        "gastar GPU en un run identico al viejo."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


---
### Paso 2b - Pool `all_lateral` (3 semillas)

Baseline: `mean_teacher_all_lateral`, 0.8602 +/- 0.0058 con 3 semillas. Es el mejor resultado de la tesis.


In [ ]:
# === UNM Mean Teacher all_lateral, batch_size_unlab=4, seed 0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_all_lateral_unlab4"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5

# --- LA UNICA DIFERENCIA con mean_teacher_all_lateral ---
# Por defecto el codigo usa batch_size // 4 = 1 frame no etiquetado por paso.
# Con 4, el modelo ve cuatro veces mas frames del pool por epoca.
# El batch etiquetado NO se toca: sigue en 5 items (30 imagenes), para que la
# comparacion contra el baseline sea limpia.
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5, "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4, "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05, "lambda_u debe igualar al del baseline"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_all_lateral/images", "pool equivocado en esta celda"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    # guardian en caliente: el loader debe tener 4 de verdad
    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando para no "
        "gastar GPU en un run identico al viejo."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM Mean Teacher all_lateral, batch_size_unlab=4, seed 1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_all_lateral_unlab4"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5

# --- LA UNICA DIFERENCIA con mean_teacher_all_lateral ---
# Por defecto el codigo usa batch_size // 4 = 1 frame no etiquetado por paso.
# Con 4, el modelo ve cuatro veces mas frames del pool por epoca.
# El batch etiquetado NO se toca: sigue en 5 items (30 imagenes), para que la
# comparacion contra el baseline sea limpia.
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5, "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4, "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05, "lambda_u debe igualar al del baseline"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_all_lateral/images", "pool equivocado en esta celda"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    # guardian en caliente: el loader debe tener 4 de verdad
    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando para no "
        "gastar GPU en un run identico al viejo."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM Mean Teacher all_lateral, batch_size_unlab=4, seed 2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_all_lateral_unlab4"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5

# --- LA UNICA DIFERENCIA con mean_teacher_all_lateral ---
# Por defecto el codigo usa batch_size // 4 = 1 frame no etiquetado por paso.
# Con 4, el modelo ve cuatro veces mas frames del pool por epoca.
# El batch etiquetado NO se toca: sigue en 5 items (30 imagenes), para que la
# comparacion contra el baseline sea limpia.
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5, "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4, "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05, "lambda_u debe igualar al del baseline"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_all_lateral/images", "pool equivocado en esta celda"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    # guardian en caliente: el loader debe tener 4 de verdad
    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando para no "
        "gastar GPU en un run identico al viejo."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


---
### Paso 3 - Resumen de los dos pools


In [ ]:
# === RESUMEN (solo lectura, no entrena nada) ===
import os, csv, statistics

BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3"

def f1_de(run_dir):
    vals = []
    if not os.path.isdir(run_dir):
        return vals
    for seed in sorted(os.listdir(run_dir)):
        p = os.path.join(run_dir, seed, "test_metrics.csv")
        if not os.path.isfile(p):
            continue
        for row in csv.reader(open(p)):
            if row and row[0] == "sample_mean_f1":
                vals.append(float(row[1]))
    return vals

def resumen(nombre, v):
    if not v:
        print("  %-34s sin runs todavia" % nombre)
        return None, None
    m = statistics.mean(v)
    s = statistics.stdev(v) if len(v) > 1 else 0.0
    print("  %-34s n=%d  F1 = %.4f +/- %.4f   %s" % (nombre, len(v), m, s, [round(x, 4) for x in v]))
    return m, s

BLOQUES = [
    ("POOL r10_max0 (3.937 frames)",
     os.path.join(BASE, "runs_final_v1", "mean_teacher_r10"),
     os.path.join(BASE, "runs_batch_unlab", "mt_r10_unlab4")),
    ("POOL all_lateral (74.774 frames)",
     os.path.join(BASE, "runs_final_v1", "mean_teacher_all_lateral"),
     os.path.join(BASE, "runs_batch_unlab", "mt_all_lateral_unlab4")),
]

print("Batch etiquetado = 5 items (30 imagenes) en todos los casos.")
print("Lo unico que cambia es el batch NO etiquetado: 1 -> 4.")
print()

for titulo, dir_viejo, dir_nuevo in BLOQUES:
    print("=" * 66)
    print(titulo)
    print("=" * 66)
    viejo = f1_de(dir_viejo)
    nuevo = f1_de(dir_nuevo)
    m_v, s_v = resumen("ACTUAL  (1 no etiquetado por paso)", viejo)
    m_n, s_n = resumen("NUEVO   (4 no etiquetados por paso)", nuevo)
    if m_v is not None and m_n is not None:
        d = m_n - m_v
        ruido = max(s_v or 0.0, 0.006)
        print()
        print("  Diferencia: %+.4f     (ruido entre semillas del baseline: %.4f)" % (d, s_v or 0.0))
        if abs(d) < ruido:
            print("  -> Dentro del ruido. No se puede concluir. Mirar el signo semilla a semilla.")
        elif d > 0:
            print("  -> Por encima del ruido y POSITIVO: alimentar mas el pool SI ayuda.")
        else:
            print("  -> Por encima del ruido y NEGATIVO: inesperado, revisar antes de concluir.")
    print()

print("Nota: si el efecto aparece en all_lateral y no en r10, encaja con la idea de que")
print("lo que faltaba eran frames NUEVOS, no repetir los mismos. Si aparece en los dos,")
print("lo que faltaba era senal de la perdida SSL por paso, no cobertura del pool.")


---

## Paso 4 - Por que empeora `all_lateral`: el control que lo decide

**Resultado del paso 2b:** alimentar el pool grande **empeora** el F1
(0,8602 -> 0,8431; las tres semillas negativas). En `r10` no pasa nada (-0,0008).

Hay dos explicaciones posibles y este control las separa:

- **(a) La perdida de consistencia.** Con 4 frames por paso la perdida pesa de verdad, y
  `all_lateral` incluye frames lejanos a la distribucion etiquetada. Es lo que predice
  Oliver 2018: datos no etiquetados de otra distribucion pueden empeorar respecto a no
  usar ninguno.
- **(b) BatchNorm.** El forward de los no etiquetados mueve las estadisticas de
  normalizacion. Con 4 frames por paso, las mueve con 4x mas frames de esa otra
  distribucion. Ya se sabe que BatchNorm aporta el 44 % de la ganancia de SSL aqui.

**El control**: `all_lateral` con `batch_size_unlab=4` y **`lambda_u = 0`**. Con la
perdida apagada, el unico canal que queda es BatchNorm.

Completa un 2x2 del que ya existen tres celdas:

| | no-etiq = 1 | no-etiq = 4 |
|---|---|---|
| **lambda_u = 0,05** | 0,8602 (n=3) | 0,8431 (n=3) |
| **lambda_u = 0** | 0,8613 (n=3) | **FALTA** |

- Si el nuevo valor **se queda en ~0,861** -> el dano lo causa la **perdida**.
- Si **cae a ~0,843** -> el dano lo causa **BatchNorm**.

⚠️ Verificado antes de escribir estas celdas: los configs de `control_lambda0_all_lateral`
y `mt_all_lateral_unlab4` difieren **solo** en `lambda_u` y `batch_size_unlab`. El 2x2
esta limpio.

Tres runs, ~30 min cada uno.


In [ ]:
# === UNM MT all_lateral, batch_size_unlab=4, lambda_u=0 (CONTROL), seed 0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_all_lateral_unlab4_lambda0"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]       = 5
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5,            "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4,      "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.0,     "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 15,     "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_all_lateral/images", "pool equivocado"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM MT all_lateral, batch_size_unlab=4, lambda_u=0 (CONTROL), seed 1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_all_lateral_unlab4_lambda0"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]       = 5
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5,            "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4,      "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.0,     "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 15,     "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_all_lateral/images", "pool equivocado"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM MT all_lateral, batch_size_unlab=4, lambda_u=0 (CONTROL), seed 2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_all_lateral_unlab4_lambda0"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]       = 5
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5,            "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4,      "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.0,     "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 15,     "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_all_lateral/images", "pool equivocado"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


---
### Resumen del 2x2


In [ ]:
# === RESUMEN 2x2 (solo lectura) - POR SEMILLA, no solo medias ===
import os, csv, statistics

BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3"

def f1(d):
    o = {}
    if not os.path.isdir(d):
        return o
    for s in sorted(os.listdir(d)):
        p = os.path.join(d, s, "test_metrics.csv")
        if not os.path.isfile(p):
            continue
        for row in csv.reader(open(p)):
            if row and row[0] == "sample_mean_f1":
                o[s] = float(row[1])
    return o

C = {
    ("0.05", "1"): os.path.join(BASE, "runs_final_v1", "mean_teacher_all_lateral"),
    ("0.05", "4"): os.path.join(BASE, "runs_batch_unlab", "mt_all_lateral_unlab4"),
    ("0",    "1"): os.path.join(BASE, "runs_control_lambda0", "control_lambda0_all_lateral"),
    ("0",    "4"): os.path.join(BASE, "runs_batch_unlab", "mt_all_lateral_unlab4_lambda0"),
}
D = {k: f1(v) for k, v in C.items()}

print("POOL all_lateral. Supervisado de referencia: 0.8015")
print()
print("MEDIAS")
print("                 no-etiq=1        no-etiq=4")
for lam in ["0.05", "0"]:
    fila = "  lambda_u=%-5s  " % lam
    for bu in ["1", "4"]:
        d = D[(lam, bu)]
        fila += ("%.4f (n=%d)   " % (statistics.mean(d.values()), len(d))) if d else "  pendiente    "
    print(fila)

print()
print("EFECTO DE 1 -> 4, SEMILLA A SEMILLA   (lo que de verdad hay que mirar)")
print()
for lam in ["0.05", "0"]:
    A, B = D[(lam, "1")], D[(lam, "4")]
    com = sorted(set(A) & set(B))
    if not com:
        print("  lambda_u=%s : pendiente" % lam); continue
    dif = [B[s] - A[s] for s in com]
    print("  lambda_u=%-5s  %s" % (lam, "  ".join("%s %+.4f" % (s.replace("seed_", "s"), d)
                                                   for s, d in zip(com, dif))))
    print("                 media %+.4f   sd de las diferencias %.4f   negativas %d/%d"
          % (statistics.mean(dif), statistics.stdev(dif) if len(dif) > 1 else 0.0,
             sum(1 for d in dif if d < 0), len(dif)))
    peor = min(zip(com, dif), key=lambda x: x[1])
    resto = [d for s, d in zip(com, dif) if s != peor[0]]
    if resto and abs(peor[1]) > 3 * max(abs(min(resto)), abs(max(resto)), 1e-9):
        print("                 AVISO: %s domina. Sin ella la media seria %+.4f"
              % (peor[0], statistics.mean(resto)))
    print()

print("COMO LEERLO - no mirar solo la media:")
print("  * Si TODAS las semillas van en el mismo sentido -> efecto real.")
print("  * Si UNA se hunde y las demas apenas se mueven -> INESTABILIDAD, no un")
print("    desplazamiento. Con 3 semillas no se distingue: hacen falta mas.")
print("  * sd entre semillas de los baselines ~0.006. Por debajo de eso, ruido.")
print()
print("Solo se puede atribuir la causa (perdida vs BatchNorm) si el patron es")
print("consistente en las semillas. Una media no basta.")


---

## Paso 5 (opcional) - Completar a 5 semillas

La caida de `all_lateral` es de -0,0171 de media, **pero la semilla 0 aporta -0,0373**.
Sin ella la media seria -0,0070, en el limite del ruido. El signo es consistente en las
tres; la magnitud no.

Para pasar de 3 a 5 semillas **pareadas** hacen falta cuatro runs: dos de la condicion
nueva y dos del baseline, porque `mean_teacher_all_lateral` tambien tiene solo 3.

Hazlo solo si despues del paso 4 quieres cerrar la magnitud. Para saber **por que** pasa,
el paso 4 es suficiente.


In [ ]:
# === UNM MT all_lateral, batch_size_unlab=4, lambda_u=0.05, seed 3 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_all_lateral_unlab4"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]       = 5
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5,            "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4,      "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05,     "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 15,     "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_all_lateral/images", "pool equivocado"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM MT all_lateral, batch_size_unlab=4, lambda_u=0.05, seed 4 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_all_lateral_unlab4"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]       = 5
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5,            "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4,      "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05,     "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 15,     "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_all_lateral/images", "pool equivocado"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM MT all_lateral BASELINE (no-etiq=1), seed 3  [completar a 5 semillas] ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
# SIN batch_size_unlab: cae en el valor por defecto (batch_size // 4 = 1),
# que es lo que usaron las semillas 0, 1 y 2 de este mismo run.

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert "batch_size_unlab" not in cfg or cfg["batch_size_unlab"] in (None, 1), (
        "Esta celda replica el baseline: NO debe llevar batch_size_unlab=4"
    )
    assert cfg["lambda_u"] == 0.05, "lambda_u debe igualar al de las semillas 0-2"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 1, (
        "FALLO: el baseline debe usar 1 frame no etiquetado por paso."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM MT all_lateral BASELINE (no-etiq=1), seed 4  [completar a 5 semillas] ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
# SIN batch_size_unlab: cae en el valor por defecto (batch_size // 4 = 1),
# que es lo que usaron las semillas 0, 1 y 2 de este mismo run.

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert "batch_size_unlab" not in cfg or cfg["batch_size_unlab"] in (None, 1), (
        "Esta celda replica el baseline: NO debe llevar batch_size_unlab=4"
    )
    assert cfg["lambda_u"] == 0.05, "lambda_u debe igualar al de las semillas 0-2"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 1, (
        "FALLO: el baseline debe usar 1 frame no etiquetado por paso."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


---

## Paso 6 - ¿Es el contenido de los frames, o que hay demasiados?

Cuando `all_lateral` empeora, hay **dos** cosas que cambiaron a la vez respecto a `r10`,
y no se pueden separar mirando solo esos dos:

| | `r10` | `all_lateral` |
|---|---|---|
| Cuantos frames tiene | 3.937 | **74.774** |
| Como son | **parecidos** a los anotados (a +/-10 posiciones, o sea a un tercio de segundo) | **variados**: de todo el video, antes y despues de tragar |

O sea: `all_lateral` es a la vez **mas grande** y **mas variado**. Cuando falla, no
sabemos cual de las dos cosas tiene la culpa.

**`std_matched_r10` es la combinacion que falta.** Es un pool de **3.938 frames**
—pequeno, como `r10`— pero elegidos **al azar de todo el video** —variados, como
`all_lateral`—:

| | Pocos frames (3.938) | Muchos frames (74.774) |
|---|---|---|
| **Parecidos** a los anotados | `r10` ✅ hecho | — |
| **Variados** | **`std_matched_r10`** ← esto | `all_lateral` ✅ hecho |

Deja fijo el tamano y cambia solo el tipo de frame. Eso es lo que aisla la causa:

- Si **tambien empeora** -> la culpa es del **contenido**. Frames variados, poco parecidos
  a los 218 anotados. Es exactamente lo que predice Oliver 2018.
- Si **no empeora** -> la culpa **no** es el contenido, sino el **tamano**: en
  `all_lateral` el modelo no vuelve a ver ningun frame nunca, y quiza necesita repetirlos
  para estabilizarse.

⚠️ Verificado antes de escribir estas celdas: los configs de `mean_teacher_std_matched_r10`
y `mean_teacher_r10` difieren **solo** en `unlabeled_subdir`. La comparacion es limpia.

**Baseline con el que se compara**: `mean_teacher_std_matched_r10`, 3 semillas,
**0,8572**.

Tres runs, ~30 min cada uno.


In [ ]:
# === UNM MT std_matched_r10, batch_size_unlab=4, seed 0 ===
# Pool PEQUENO (3.938, como r10) pero de frames VARIADOS (aleatorios de todo
# el video, como all_lateral). Separa "contenido" de "tamano".
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_std_matched_r10_unlab4"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]       = 5
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5,       "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4, "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05,      "lambda_u debe igualar al del baseline"
    assert cfg["unlabeled_subdir"] == "unlabeling_std_matched_r10/images", "pool equivocado"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM MT std_matched_r10, batch_size_unlab=4, seed 1 ===
# Pool PEQUENO (3.938, como r10) pero de frames VARIADOS (aleatorios de todo
# el video, como all_lateral). Separa "contenido" de "tamano".
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_std_matched_r10_unlab4"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]       = 5
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5,       "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4, "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05,      "lambda_u debe igualar al del baseline"
    assert cfg["unlabeled_subdir"] == "unlabeling_std_matched_r10/images", "pool equivocado"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM MT std_matched_r10, batch_size_unlab=4, seed 2 ===
# Pool PEQUENO (3.938, como r10) pero de frames VARIADOS (aleatorios de todo
# el video, como all_lateral). Separa "contenido" de "tamano".
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_std_matched_r10_unlab4"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]       = 5
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5,       "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4, "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.05,      "lambda_u debe igualar al del baseline"
    assert cfg["unlabeled_subdir"] == "unlabeling_std_matched_r10/images", "pool equivocado"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


---
### Resumen: contenido o tamano


In [ ]:
# === RESUMEN: contenido o tamano? (solo lectura) ===
import os, csv, statistics

BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3"

def f1(d):
    o = {}
    if not os.path.isdir(d):
        return o
    for s in sorted(os.listdir(d)):
        p = os.path.join(d, s, "test_metrics.csv")
        if not os.path.isfile(p):
            continue
        for row in csv.reader(open(p)):
            if row and row[0] == "sample_mean_f1":
                o[s] = float(row[1])
    return o

FILAS = [
    ("r10          (3.938, parecidos)",
     os.path.join(BASE, "runs_final_v1", "mean_teacher_r10"),
     os.path.join(BASE, "runs_batch_unlab", "mt_r10_unlab4")),
    ("std_matched  (3.938, variados) ",
     os.path.join(BASE, "runs_final_v1", "mean_teacher_std_matched_r10"),
     os.path.join(BASE, "runs_batch_unlab", "mt_std_matched_r10_unlab4")),
    ("all_lateral  (74.774, variados)",
     os.path.join(BASE, "runs_final_v1", "mean_teacher_all_lateral"),
     os.path.join(BASE, "runs_batch_unlab", "mt_all_lateral_unlab4")),
]

print("Supervisado de referencia: 0.8015")
print()
print("POOL                              con 1        con 4        Δ")
print("-" * 66)
deltas = {}
for nombre, dv, dn in FILAS:
    V, N = f1(dv), f1(dn)
    if V and N:
        comunes = sorted(set(V) & set(N))
        d = statistics.mean(N[s] - V[s] for s in comunes) if comunes else None
        deltas[nombre.split()[0]] = d
        print("%s  %.4f      %.4f      %+.4f  (%d semillas pareadas)"
              % (nombre, statistics.mean(V.values()), statistics.mean(N.values()), d, len(comunes)))
    elif V:
        print("%s  %.4f      pendiente" % (nombre, statistics.mean(V.values())))
    else:
        print("%s  pendiente" % nombre)

print()
if "std_matched" in deltas and deltas["std_matched"] is not None:
    d_std = deltas["std_matched"]
    print("Lectura:")
    if d_std < -0.006:
        print("  std_matched TAMBIEN empeora, y es un pool PEQUENO.")
        print("  -> La causa es el CONTENIDO: frames variados, poco parecidos a los 218")
        print("     anotados. No es el tamano ni la falta de repeticion.")
        print("     Es lo que predice Oliver 2018.")
    elif abs(d_std) <= 0.006:
        print("  std_matched NO empeora, aunque sus frames son igual de variados.")
        print("  -> La causa NO es solo el contenido. Apunta al TAMANO del pool:")
        print("     en all_lateral el modelo no repite ningun frame nunca.")
    else:
        print("  std_matched MEJORA. Resultado inesperado: revisar antes de concluir.")
    print()
    print("  Recordatorio: sd entre semillas de los baselines ~0.006.")
    print("  Por debajo de eso no se distingue del ruido.")
else:
    print("Corre las tres celdas de std_matched antes de interpretar.")


---

## Paso 7 - Mas semillas donde esta la duda

El 2x2 con 3 semillas no distingue **desplazamiento** de **inestabilidad**: en
cada condicion **una sola semilla** se hunde y las otras apenas se mueven, y es
una semilla distinta en cada una. Con la media parece un efecto; semilla a
semilla parece un run que descarrila de vez en cuando.

Lo que falta no es precision en la media: es saber **cada cuanto descarrila**.
Eso solo se ve con mas semillas en las dos condiciones de `no-etiq=4`.

Cuatro runs, ~30 min cada uno.


In [ ]:
# === UNM MT all_lateral, batch_size_unlab=4, lambda_u=0 (CONTROL), seed 3 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_all_lateral_unlab4_lambda0"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]       = 5
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5,            "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4,      "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.0,     "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 15,     "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_all_lateral/images", "pool equivocado"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM MT all_lateral, batch_size_unlab=4, lambda_u=0 (CONTROL), seed 4 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mt_all_lateral_unlab4_lambda0"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_batch_unlab/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]       = 5
cfg["batch_size_unlab"] = 4

cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
# run_summary.txt solo se escribe cuando el entrenamiento TERMINA.
# Sin el, un best_model.pt es de un run cortado a medias (crash de CUDA,
# desconexion de Colab...) y NO se puede evaluar como si fuera valido.
_has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

_skip_all  = _has_best and _has_metrics and _has_summary
_eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

if _has_best and not _has_summary:
    print("AVISO: hay best_model.pt pero NO run_summary.txt.")
    print("       El entrenamiento anterior se corto a medias. Se reentrena desde cero.")

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["batch_size"] == 5,            "el batch etiquetado debe seguir en 5"
    assert cfg["batch_size_unlab"] == 4,      "esta celda exige batch_size_unlab=4"
    assert cfg["lambda_u"] == 0.0,     "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 15,     "semi_start no coincide con el dataset"
    assert cfg["unlabeled_subdir"] == "unlabeling_all_lateral/images", "pool equivocado"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    assert loaders["unlabeled_loader"].batch_size == 4, (
        "FALLO: el DataLoader no recibio batch_size_unlab=4. Abortando."
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)
